# Unit 6: 高级 RNN 技术

## 学习目标
- 掌握双向 RNN (Bidirectional RNN)
- 理解多层堆叠 RNN 的设计
- 学会在 RNN 中使用 Dropout 正则化
- 理解 PackedSequence 处理变长序列
- 入门 Attention 机制

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

torch.manual_seed(42)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## 6.1 双向 RNN (Bidirectional RNN)

标准 RNN 只能利用**过去的信息**。双向 RNN 同时从两个方向处理序列：
- **前向**: 从左到右，捕获上文
- **后向**: 从右到左，捕获下文
- **合并**: 拼接或求和两个方向的隐藏状态

应用场景：文本分类、命名实体识别（需要上下文信息）。

注意：**不适用于文本生成**（生成时没有"未来"信息）。

In [ ]:
# 双向 LSTM
input_size, hidden_size = 8, 16
bilstm = nn.LSTM(
    input_size=input_size,
    hidden_size=hidden_size,
    num_layers=1,
    batch_first=True,
    bidirectional=True, # 双向 LSTM
)

x = torch.randn(3, 10, input_size)  # (batch, seq_len, features)
output, (h_n, c_n) = bilstm(x)
# output: 最后一层的输出, (batch, seq_len, hidden_size * 2)
# h_n: (2*1, 3, 16) 双向=2层, 每层的输出维度为 16

print(f"输入: {x.shape}")
print(f"输出: {output.shape}")  # hidden_size * 2 = 32
print(f"h_n:   {h_n.shape}")    # (2*1, 3, 16) 双向=2层
print(f"\n因为 bidirectional=True, 输出维度变为 hidden_size * 2")
print(f"h_n[0] = 前向最后一步, h_n[1] = 后向最后一步")

In [ ]:
# 双向 RNN 在序列标注任务中的应用
class BiLSTM_Tagger(nn.Module):
    """双向 LSTM 用于序列标注 (如 POS tagging)"""
    def __init__(self, vocab_size, embed_dim, hidden_size, num_tags):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.bilstm = nn.LSTM(embed_dim, hidden_size,
                              bidirectional=True, batch_first=True)
        self.classifier = nn.Linear(hidden_size * 2, num_tags)

    def forward(self, x):
        emb = self.embedding(x)               # (B, T, E)
        lstm_out, _ = self.bilstm(emb)        # (B, T, 2*H)
        logits = self.classifier(lstm_out)    # (B, T, num_tags)
        return logits

# 测试
model = BiLSTM_Tagger(vocab_size=100, embed_dim=32, hidden_size=64, num_tags=5)
x_demo = torch.randint(0, 100, (2, 8))  # 2句话，每句8个token
logits = model(x_demo)
print(f"输入序列: {x_demo.shape}")
print(f"每个 token 的标签 logits: {logits.shape}")

## 6.2 多层堆叠 RNN

增加 RNN 层数可以学习更抽象的特征表示：
- 第 1 层: 低级模式（字符组合）
- 第 2 层: 中级模式（词级语义）
- 第 3+ 层: 高级抽象（句法结构）

`num_layers=N` 即可实现堆叠。注意层间可以加 Dropout。

In [ ]:
# 对比不同层数的 RNN
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

for num_layers in [1, 2, 3, 4]:
    lstm = nn.LSTM(128, 256, num_layers=num_layers, batch_first=True)
    print(f"LSTM (layers={num_layers}): {count_parameters(lstm):,} params")



In [ ]:
# 多层 LSTM 的数据流
deep_lstm = nn.LSTM(
    input_size=16,
    hidden_size=32,
    num_layers=3,
    batch_first=True,
    dropout=0.3,  # 层间 dropout (仅 num_layers > 1 时生效)
)

x = torch.randn(4, 10, 16)
output, (h_n, c_n) = deep_lstm(x)
print(f"\n输出: {output.shape}")  # (4, 10, 32)
print(f"h_n: {h_n.shape}")        # (3, 4, 32) 每层一个隐藏状态
print(f"h_n[0] 是第1层的, h_n[1] 是第2层的, h_n[2] 是第3层的")

## 6.3 RNN 中的 Dropout 策略

RNN 的 Dropout 有两种位置：

1. **层间 Dropout** (`dropout` 参数): 在 RNN 层之间应用（需 `num_layers > 1`）
2. **嵌入后 Dropout**: 在 embedding 后加 `nn.Dropout`
3. **输出 Dropout**: 在 RNN 输出后加 `nn.Dropout`

**注意**：RNN 的 dropout 不会应用在时间维度上（避免破坏时序依赖）。

In [ ]:
# Dropout 对比实验
class RNNWithDropout(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, output_size,
                 dropout_embed=0.3, dropout_output=0.3, dropout_rnn=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout_embed = nn.Dropout(dropout_embed)       # 嵌入层后
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers=2,
                           batch_first=True, dropout=dropout_rnn)  # 层间
        self.dropout_output = nn.Dropout(dropout_output)     # 输出层前
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout_embed(x)
        x, _ = self.lstm(x)
        x = self.dropout_output(x[:, -1, :])  # 取最后一步
        return self.fc(x)

model = RNNWithDropout(100, 64, 128, 2)
print(f"训练模式时 Dropout 生效，测试模式时关闭")
print(f"model.train()  → Dropout 随机置零神经元")
print(f"model.eval()   → Dropout 关闭，保留全部权重")

## 6.4 PackedSequence: 高效处理变长序列

当 batch 中序列长度差异很大时，直接 padding 会浪费大量计算。

`PackedSequence` 通过跳过 padding 位置，只在有效 token 上计算 RNN。

**流程**: `pad_sequence → pack_padded_sequence → RNN → pad_packed_sequence`

| 值 | 行为 | 适用场景 |
| :--- | :--- | :--- |
| `True` (默认) | 强制检查输入是否已按长度降序排序。若未排序，直接抛出 `RuntimeError` | 你已手动排好序，想省去内部排序开销以获得极致性能 |
| `False` | 自动排序：内部先对序列按长度降序排序，处理完后再恢复原始顺序 | 输入顺序不确定或不想手动排序（推荐日常使用） |



历史上，CuDNN 的 RNN 实现要求输入必须严格按长度降序排列，否则无法正确计算。PyTorch 早期版本因此将 enforce_sorted=True 设为默认值，把排序责任交给用户。后来 PyTorch 在内部实现了自动排序+还原逻辑，引入了 enforce_sorted=False，但为了向后兼容，默认值仍保持为 True。


| 对比项 | Padded 模式输入 | Packed 模式输入 |
| :--- | :--- | :--- |
| 返回类型 | `Tensor` | `PackedSequence` ⚠️ |
| 形状 | `(batch, max_len, hidden)` | `data: (total_valid_tokens, hidden)` <br> `batch_sizes: [3,3,3,2,2,1,1]` |
| Padding位置 | 包含全0向量（或无意义值） | 不存在，只包含有效token的输出 |
| 后续处理 | 可直接使用 | 必须先 `pad_packed_sequence()` 才能变回普通Tensor |

核心区别： Packed 模式下 LSTM 的输出仍然是压缩状态。这是为了支持多层 RNN 的链式调用（上一层的 packed 输出可以直接作为下一层的 packed 输入），避免中间反复 pack/unpack 的开销。

| 对比项 | Padded 模式输入 | Packed 模式输入 |
| :--- | :--- | :--- |
| 形状 | `(num_layers, batch, hidden)` | `(num_layers, batch, hidden)` ✅ 相同 |
| 数值正确性 | 每个序列最后时间步的状态 | 每个序列最后有效时间步的状态 ✅ 正确 |
| ⚠️ 批次顺序 | 原始输入顺序 `[Seq0, Seq1, Seq2]` | 排序后的顺序 `[Seq0, Seq2, Seq1]` |

⚠️ 最大陷阱： h_n, c_n 的批次维度是按长度降序排列的，不是原始输入顺序！如果你直接用它们做下游任务（如分类、解码），结果会错位。

In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

# === 原有代码保持不变 ===
seqs = [
    torch.randn(7, 4),   # 长度 7
    torch.randn(3, 4),   # 长度 3
    torch.randn(5, 4),   # 长度 5
]
lengths = torch.tensor([7, 3, 5])

padded = pad_sequence(seqs, batch_first=True)
print(f"Padded shape: {padded.shape}")  # (3, 7, 4)

packed = pack_padded_sequence(padded, lengths, batch_first=True, enforce_sorted=False)
print(f"\nPackedSequence:")
print(f"  data shape: {packed.data.shape}")       # (15, 4)
print(f"  batch_sizes: {packed.batch_sizes}")      # [3, 3, 3, 2, 2, 1, 1]
print(f"  sorted_indices: {packed.sorted_indices}") # [0, 2, 1] ← 新增观察点

lstm = nn.LSTM(4, 8, batch_first=True)
packed_out, (h_n, c_n) = lstm(packed)
print(f"\nRNN output (packed) data shape: {packed_out.data.shape}")  # (15, 8)

unpacked, out_lengths = pad_packed_sequence(packed_out, batch_first=True)
print(f"Unpacked shape: {unpacked.shape}")  # (3, 7, 8)

print(f"\n优势: 只计算了 {packed.data.shape[0]} 个 token 而非 {padded.shape[0]*padded.shape[1]}")

# ============================================================
# 🔽 以下为关键补充内容 🔽
# ============================================================

# ✅ 补充1: 恢复 h_n, c_n 到原始批次顺序
# packed模式下 h_n/c_n 是按长度降序排列的 [Seq0, Seq2, Seq1]
# 必须用 sorted_indices 的逆映射才能对齐回原始输入 [Seq0, Seq1, Seq2]
_, unsort_idx = packed.sorted_indices.sort()
h_n_restored = h_n[:, unsort_idx, :]
c_n_restored = c_n[:, unsort_idx, :]
print(f"\nh_n 原始顺序恢复: {h_n_restored.shape}")  # (1, 3, 8)

# ✅ 补充2: 验证 unpacked 输出与 h_n 的一致性
# 对于每个序列，unpacked输出的最后一个有效时间步 == h_n
for i in range(len(seqs)):
    last_valid = out_lengths[i] - 1
    out_last = unpacked[i, last_valid, :]   # unpacked已自动恢复原始顺序
    h_last = h_n_restored[0, i, :]          # 取第0层
    diff = (out_last - h_last).abs().max().item()
    print(f"  Seq{i}: max|unpacked[last] - h_n| = {diff:.2e}")  # 应 ≈ 1e-7

# ✅ 补充3: 确认 padding 区域确实为零
padding_region = unpacked[1, 3:, :]  # Seq1 长度=3, 索引3~6应为全0
print(f"\nSeq1 padding区域全零验证: {(padding_region == 0).all().item()}")  # True

# ✅ 补充4: 与 padded 模式的结果对比（证明数值等价）
lstm.eval()  # 关闭dropout等随机行为
with torch.no_grad():
    # Padded 模式直接跑
    padded_out_direct, (h_n_direct, _) = lstm(padded)
    
    # 比较两者在有效区域内的输出是否一致
    max_diff = 0.0
    for i in range(len(seqs)):
        L = lengths[i]
        d = (unpacked[i, :L, :] - padded_out_direct[i, :L, :]).abs().max().item()
        max_diff = max(max_diff, d)
    print(f"\nPacked vs Padded 有效区域最大差异: {max_diff:.2e}")  # 应 ≈ 1e-6

## 6.5 Attention 机制入门

Attention 解决了 RNN 的"信息瓶颈"问题：
- RNN 将整个序列压缩到最后的隐藏状态中
- Attention 让解码器可以**直接访问**编码器所有时间步的输出

### Bahdanau Attention (Additive)

$$e_{t,i} = v^T \tanh(W_h h_i + W_s s_{t-1})$$
$$\alpha_{t,i} = \text{softmax}(e_t)_i$$
$$c_t = \sum_i \alpha_{t,i} h_i$$

其中 $h_i$ 是编码器输出，$s_t$ 是解码器状态，$\alpha$ 是注意力权重。

这段代码实现了一个经典的 Bahdanau Attention（加性注意力） 机制。这是序列到序列（Seq2Seq）模型中最早被提出的注意力机制之一，用于解决长序列信息丢失问题。

In [ ]:
# 简单 Attention 实现
class SimpleAttention(nn.Module):
    """Bahdanau Attention"""
    def __init__(self, encoder_hidden, decoder_hidden, attention_dim):
        super().__init__()
        self.W_h = nn.Linear(encoder_hidden, attention_dim, bias=False)
        self.W_s = nn.Linear(decoder_hidden, attention_dim)
        self.v = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, encoder_outputs, decoder_hidden):
        """
        Args:
            encoder_outputs: (batch, src_len, enc_hidden)
            decoder_hidden:  (batch, dec_hidden)
        Returns:
            context:  (batch, enc_hidden) 注意力加权上下文向量
            attention_weights: (batch, src_len) 注意力分布
        """
        # encoder_outputs: (B, S, H_enc)
        # decoder_hidden:  (B, H_dec) → unsqueeze to (B, 1, H_dec)
        dec_hidden = decoder_hidden.unsqueeze(1)  # (B, 1, H_dec)

        # 计算能量分数
        energy = self.v(torch.tanh(
            self.W_h(encoder_outputs) + self.W_s(dec_hidden)
        )).squeeze(-1)  # (B, S)

        # Softmax 得到注意力权重
        attention_weights = F.softmax(energy, dim=1)  # (B, S)

        # 加权求和得到上下文向量
        context = torch.bmm(
            attention_weights.unsqueeze(1),   # (B, 1, S)
            encoder_outputs                   # (B, S, H_enc)
        ).squeeze(1)  # (B, H_enc)

        return context, attention_weights

# 测试 Attention
batch, src_len, enc_hidden, dec_hidden, attn_dim = 2, 5, 64, 128, 32

attention = SimpleAttention(enc_hidden, dec_hidden, attn_dim)
enc_out = torch.randn(batch, src_len, enc_hidden)
dec_hid = torch.randn(batch, dec_hidden)

context, weights = attention(enc_out, dec_hid)
print(f"Encoder outputs: {enc_out.shape}")
print(f"Context vector:  {context.shape}")
print(f"Attention weights: {weights.shape}")
print(f"\nAttention 权重 (每个源位置的重要性):\n{weights}")

In [ ]:
# 可视化 Attention 权重
def visualize_attention(attention_weights, src_tokens):
    fig, ax = plt.subplots(figsize=(8, 2))
    im = ax.imshow(attention_weights.detach().unsqueeze(0).numpy(), cmap='Blues', aspect='auto')
    ax.set_xticks(range(len(src_tokens)))
    ax.set_xticklabels(src_tokens)
    ax.set_yticks([])
    ax.set_title("Attention Weights")
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.show()

# 模拟一个简单的翻译示例
src = ["I", "love", "deep", "learning"]
dummy_enc_out = torch.randn(1, 4, 64)
dummy_dec_hid = torch.randn(1, 128)

attention = SimpleAttention(64, 128, 32)
_, attn_w = attention(dummy_enc_out, dummy_dec_hid)
visualize_attention(attn_w, src)
print("颜色越深表示模型对该位置的关注度越高")

## 要点总结

| 技术 | 用法 | 适用场景 |
|------|------|----------|
| 双向 RNN | `bidirectional=True` | 分类、标注（需要看全文） |
| 多层堆叠 | `num_layers=N` | 复杂模式学习 |
| Dropout | `dropout=` 参数 / `nn.Dropout` | 防止过拟合 |
| PackedSequence | `pack_padded_sequence` | 变长序列高效处理 |
| Attention | 权重加权求和 | 长序列、翻译、需要可解释性 |

### 练习
1. 用 PackedSequence 加速一个变长文本分类任务
2. 对比单向 vs 双向 LSTM 在情感分类上的准确率
3. 实现 Dot-Product Attention 与 Bahdanau Attention 对比